# 🏍️ ServisPro: Sistem Pakar Rekomendasi Servis Berkala Motor Matic
### Metode Inferensi: **Backward Chaining** (*goal-driven*)

**Nama:** Muhamad Radi Taher
**NPM:** 202343501083
**Kelas:** R6M

---

**Isi notebook:**
1. Penjelasan sistem & metode Backward Chaining
2. Basis pengetahuan (interval servis, fakta, rule, goal)
3. Mesin inferensi Backward Chaining
4. Pengujian dengan skenario demo
5. Pengujian *black box* beberapa skenario
6. **Simpan basis pengetahuan** ke file `basis_pengetahuan_servis.sav`
7. Menjalankan aplikasi web `app.py` (Streamlit)
8. Kesimpulan

> Jalankan semua cell berurutan (**Run All**). File `.sav` yang dihasilkan dipakai oleh `app.py`.

## 1. Penjelasan Sistem

Sistem pakar ini merekomendasikan **jenis servis / penggantian komponen** motor matic berdasarkan jarak tempuh (km), lama pemakaian (bulan), dan keluhan yang dirasakan pengguna.

**Backward Chaining** adalah metode inferensi yang bekerja **mundur dari kesimpulan (goal) menuju fakta**:

1. Sistem mengambil satu **goal**, misalnya *Ganti Busi*.
2. Sistem mencari rule yang **THEN**-nya goal tersebut (misal R13 dan R14, hubungan **OR**).
3. Setiap **premis (IF)** dari rule dibuktikan satu per satu (hubungan **AND**):
   - Kalau premis adalah kesimpulan rule lain, premis itu jadi **sub-goal** dan dibuktikan secara rekursif.
   - Kalau premis adalah **fakta dasar**, sistem bertanya ke pengguna.
4. Jika semua premis terpenuhi, goal **TERBUKTI**. Jika semua rule gagal, goal **TIDAK terbukti**.
5. Semua jawaban disimpan di **working memory** agar pertanyaan tidak diulang.

Kelebihannya: sistem **hanya menanyakan hal yang dibutuhkan** untuk membuktikan goal.

```
GOAL: Ganti Busi?
 └─ R14: IF indikasi_pengapian_lemah
      └─ SUB-GOAL: indikasi_pengapian_lemah?
           └─ R01: IF mesin_susah_hidup AND mesin_brebet
                ├─ FAKTA: mesin susah hidup?  -> tanya user
                └─ FAKTA: mesin brebet?       -> tanya user
```

In [1]:
import pickle
import pandas as pd

print("Library siap.")

Library siap.


## 2. Basis Pengetahuan (Knowledge Base)

### 2.1 Interval Servis Berkala
Mengacu pada **Buku Pedoman Pemilik Honda Vario 125 (PT Astra Honda Motor)** dan jadwal KPB Honda. Nilai bertanda **[bengkel]** adalah rekomendasi umum bengkel resmi karena tidak tercantum pasti di buku pedoman.

| Komponen | Interval |
|---|---|
| Oli mesin | 4.000 km (pertama 1.000 km) / ±3 bulan [bengkel] |
| Oli gardan | 8.000 km / 2 tahun |
| Busi | 8.000 km |
| Filter udara (viscous) | 16.000 km (8.000 km jika sering berdebu) |
| Servis CVT | 8.000 km [bengkel] |
| V-belt | 24.000 km (16.000 km jika CVT bergejala) |
| Injektor / throttle body | ±8.000 km [bengkel] |
| Minyak rem | 2 tahun |
| Air radiator (coolant) | 3 tahun |

In [2]:
# 1a. Parameter interval servis berkala
# Sumber utama: Buku Pedoman Pemilik Honda Vario 125 (PT Astra Honda
# Motor) dan jadwal KPB Honda (1.000 / 4.000 / 8.000 / 12.000 km).
# Nilai bertanda [bengkel] adalah rekomendasi umum bengkel resmi karena
# buku pedoman tidak mencantumkan angka pastinya.
# Ubah nilai di sini untuk menyesuaikan merek/tipe motor lain.
# ---------------------------------------------------------------------
INTERVAL = {
    "oli_mesin_km":        4000,   # ganti tiap 4.000 km (pertama di 1.000 km)
    "oli_mesin_bulan":     3,      # [bengkel] maks. ±3 bulan untuk pemakaian harian
    "oli_gardan_km":       8000,   # ganti tiap 8.000 km
    "oli_gardan_bulan":    24,     # atau tiap 2 tahun (buku pedoman)
    "busi_km":             8000,   # ganti tiap 8.000 km
    "filter_udara_km":     16000,  # saringan udara tipe viscous: ganti tiap 16.000 km
    "filter_udara_debu_km": 8000,  # lebih cepat jika sering lewat jalan berdebu
    "servis_cvt_km":       8000,   # [bengkel] bersihkan CVT & cek roller tiap 8.000 km
    "vbelt_km":            24000,  # ganti V-belt tiap 24.000 km
    "vbelt_gejala_km":     16000,  # [bengkel] ganti lebih awal jika CVT bergejala
    "injeksi_km":          8000,   # [bengkel] servis injektor/throttle body ±8.000 km
    "minyak_rem_bulan":    24,     # ganti minyak rem tiap 2 tahun (buku pedoman)
    "coolant_bulan":       36,     # ganti cairan radiator tiap 3 tahun (buku pedoman)
}

### 2.2 Parameter Numerik
Pertanyaan angka (km / bulan). Setiap parameter **cukup ditanya sekali**, lalu hasilnya dipakai oleh beberapa fakta.

In [3]:
# 1b. Parameter numerik yang ditanyakan ke pengguna (disimpan sekali)
# ---------------------------------------------------------------------
PARAMETER = {
    "km_oli_mesin":   "Berapa km sejak terakhir ganti OLI MESIN?",
    "bln_oli_mesin":  "Berapa bulan sejak terakhir ganti OLI MESIN?",
    "km_oli_gardan":  "Berapa km sejak terakhir ganti OLI GARDAN?",
    "bln_oli_gardan": "Berapa bulan sejak terakhir ganti OLI GARDAN?",
    "km_busi":        "Berapa km sejak terakhir ganti BUSI?",
    "km_filter":      "Berapa km sejak terakhir ganti FILTER UDARA?",
    "km_cvt":         "Berapa km sejak terakhir SERVIS/BERSIHKAN CVT?",
    "km_vbelt":       "Berapa km sejak terakhir ganti V-BELT?",
    "km_injeksi":     "Berapa km sejak terakhir SERVIS INJEKSI/THROTTLE BODY?",
    "bln_minyak_rem": "Berapa bulan sejak terakhir ganti MINYAK REM?",
    "bln_coolant":    "Berapa bulan sejak terakhir ganti AIR RADIATOR (coolant)?",
}

### 2.3 Fakta Dasar (*Askable Facts*)
- **numerik**: bernilai benar jika jawaban ≥ batas interval
- **ya_tidak**: kondisi fisik / keluhan yang ditanyakan langsung

In [4]:
# 1c. Fakta dasar (askable facts)
#   - tipe "numerik"  : bernilai benar jika parameter >= batas interval
#   - tipe "ya_tidak" : ditanyakan langsung ke pengguna
# ---------------------------------------------------------------------
FAKTA_DASAR = {
    # --- fakta numerik (pengecekan interval)
    "km_oli_mesin_lewat":   {"tipe": "numerik", "param": "km_oli_mesin",   "batas": "oli_mesin_km"},
    "bln_oli_mesin_lewat":  {"tipe": "numerik", "param": "bln_oli_mesin",  "batas": "oli_mesin_bulan"},
    "km_oli_gardan_lewat":  {"tipe": "numerik", "param": "km_oli_gardan",  "batas": "oli_gardan_km"},
    "bln_oli_gardan_lewat": {"tipe": "numerik", "param": "bln_oli_gardan", "batas": "oli_gardan_bulan"},
    "km_busi_lewat":        {"tipe": "numerik", "param": "km_busi",        "batas": "busi_km"},
    "km_filter_lewat":      {"tipe": "numerik", "param": "km_filter",      "batas": "filter_udara_km"},
    "km_filter_debu_lewat": {"tipe": "numerik", "param": "km_filter",      "batas": "filter_udara_debu_km"},
    "km_cvt_lewat":         {"tipe": "numerik", "param": "km_cvt",         "batas": "servis_cvt_km"},
    "km_vbelt_lewat":       {"tipe": "numerik", "param": "km_vbelt",       "batas": "vbelt_km"},
    "km_vbelt_gejala_lewat":{"tipe": "numerik", "param": "km_vbelt",       "batas": "vbelt_gejala_km"},
    "km_injeksi_lewat":     {"tipe": "numerik", "param": "km_injeksi",     "batas": "injeksi_km"},
    "bln_minyak_rem_lewat": {"tipe": "numerik", "param": "bln_minyak_rem", "batas": "minyak_rem_bulan"},
    "bln_coolant_lewat":    {"tipe": "numerik", "param": "bln_coolant",    "batas": "coolant_bulan"},

    # --- fakta ya/tidak (kondisi fisik & keluhan)
    "indikator_oli_menyala": {"tipe": "ya_tidak", "tanya": "Apakah indikator 'OIL CHANGE' di panel meter menyala?"},
    "oli_hitam_berkurang":   {"tipe": "ya_tidak", "tanya": "Saat dicek, apakah oli mesin hitam pekat atau di bawah batas minimum?"},
    "gardan_rembes":         {"tipe": "ya_tidak", "tanya": "Apakah ada rembesan oli di area gardan/roda belakang?"},
    "mesin_susah_hidup":     {"tipe": "ya_tidak", "tanya": "Apakah mesin susah dihidupkan?"},
    "mesin_brebet":          {"tipe": "ya_tidak", "tanya": "Apakah mesin brebet/tersendat saat digas?"},
    "sering_jalan_berdebu":  {"tipe": "ya_tidak", "tanya": "Apakah motor sering dipakai di jalan berdebu/proyek?"},
    "tarikan_berat":         {"tipe": "ya_tidak", "tanya": "Apakah tarikan motor terasa berat/loyo?"},
    "bunyi_cvt":             {"tipe": "ya_tidak", "tanya": "Apakah terdengar bunyi kasar/mendengung dari area CVT?"},
    "gredek":                {"tipe": "ya_tidak", "tanya": "Apakah motor bergetar (gredek) saat mulai jalan?"},
    "boros_bensin":          {"tipe": "ya_tidak", "tanya": "Apakah konsumsi bensin terasa lebih boros?"},
    "langsam_tidak_stabil":  {"tipe": "ya_tidak", "tanya": "Apakah putaran mesin saat langsam naik-turun/tidak stabil?"},
    "kampas_batas_aus":      {"tipe": "ya_tidak", "tanya": "Apakah kampas rem sudah mencapai garis/indikator batas aus (±1 mm)?"},
    "rem_berbunyi":          {"tipe": "ya_tidak", "tanya": "Apakah rem berbunyi mencicit/gesekan logam saat dipakai?"},
    "rem_kurang_pakem":      {"tipe": "ya_tidak", "tanya": "Apakah rem terasa kurang pakem/jarak pengereman lebih jauh?"},
    "minyak_rem_keruh":      {"tipe": "ya_tidak", "tanya": "Apakah minyak rem di reservoir keruh/gelap atau di bawah garis LOWER?"},
    "motor_berradiator":     {"tipe": "ya_tidak", "tanya": "Apakah motor Anda menggunakan pendingin radiator (cairan)?"},
    "mesin_overheat":        {"tipe": "ya_tidak", "tanya": "Apakah mesin sering overheat/terlalu panas?"},
    "coolant_berkurang":     {"tipe": "ya_tidak", "tanya": "Apakah air radiator di tabung cadangan sering berkurang?"},
}

### 2.4 Rule Base
Total **28 rule** berbentuk `IF premis AND premis THEN kesimpulan`. Beberapa rule dengan kesimpulan sama berarti hubungan **OR**.

| Kelompok | Rule | Kesimpulan |
|---|---|---|
| Sub-goal | R01–R05 | indikasi pengapian lemah, masalah CVT, injeksi kotor, kampas habis |
| Oli mesin | R06–R09 | ganti_oli_mesin |
| Oli gardan | R10–R12 | ganti_oli_gardan |
| Busi | R13–R14 | ganti_busi |
| Filter udara | R15–R16 | ganti_filter_udara |
| CVT | R17–R18 | servis_cvt |
| V-belt | R19–R20 | ganti_vbelt |
| Injeksi | R21–R22 | servis_injeksi |
| Kampas rem | R23–R24 | ganti_kampas_rem |
| Minyak rem | R25–R26 | ganti_minyak_rem |
| Coolant | R27–R28 | ganti_coolant |

In [5]:
# 1d. Rule base (IF premis AND premis ... THEN kesimpulan)
#   Beberapa rule dengan kesimpulan yang sama = hubungan OR.
# ---------------------------------------------------------------------
RULES = [
    # ---- Sub-goal (fakta antara)
    {"id": "R01", "if": ["mesin_susah_hidup", "mesin_brebet"], "then": "indikasi_pengapian_lemah"},
    {"id": "R02", "if": ["tarikan_berat", "bunyi_cvt"],        "then": "indikasi_masalah_cvt"},
    {"id": "R03", "if": ["gredek"],                            "then": "indikasi_masalah_cvt"},
    {"id": "R04", "if": ["boros_bensin", "langsam_tidak_stabil"], "then": "indikasi_injeksi_kotor"},
    {"id": "R05", "if": ["rem_berbunyi", "rem_kurang_pakem"],  "then": "indikasi_kampas_habis"},

    # ---- Goal: Ganti oli mesin
    {"id": "R06", "if": ["km_oli_mesin_lewat"],    "then": "ganti_oli_mesin"},
    {"id": "R07", "if": ["bln_oli_mesin_lewat"],   "then": "ganti_oli_mesin"},
    {"id": "R08", "if": ["indikator_oli_menyala"], "then": "ganti_oli_mesin"},
    {"id": "R09", "if": ["oli_hitam_berkurang"],   "then": "ganti_oli_mesin"},

    # ---- Goal: Ganti oli gardan
    {"id": "R10", "if": ["km_oli_gardan_lewat"],  "then": "ganti_oli_gardan"},
    {"id": "R11", "if": ["bln_oli_gardan_lewat"], "then": "ganti_oli_gardan"},
    {"id": "R12", "if": ["gardan_rembes"],        "then": "ganti_oli_gardan"},

    # ---- Goal: Ganti busi
    {"id": "R13", "if": ["km_busi_lewat"],            "then": "ganti_busi"},
    {"id": "R14", "if": ["indikasi_pengapian_lemah"], "then": "ganti_busi"},

    # ---- Goal: Ganti filter udara
    {"id": "R15", "if": ["km_filter_lewat"],                              "then": "ganti_filter_udara"},
    {"id": "R16", "if": ["sering_jalan_berdebu", "km_filter_debu_lewat"], "then": "ganti_filter_udara"},

    # ---- Goal: Servis/pembersihan CVT
    {"id": "R17", "if": ["km_cvt_lewat"],         "then": "servis_cvt"},
    {"id": "R18", "if": ["indikasi_masalah_cvt"], "then": "servis_cvt"},

    # ---- Goal: Ganti V-belt
    {"id": "R19", "if": ["km_vbelt_lewat"],                                "then": "ganti_vbelt"},
    {"id": "R20", "if": ["indikasi_masalah_cvt", "km_vbelt_gejala_lewat"], "then": "ganti_vbelt"},

    # ---- Goal: Servis injektor & throttle body
    {"id": "R21", "if": ["km_injeksi_lewat"],       "then": "servis_injeksi"},
    {"id": "R22", "if": ["indikasi_injeksi_kotor"], "then": "servis_injeksi"},

    # ---- Goal: Ganti kampas rem
    {"id": "R23", "if": ["kampas_batas_aus"],      "then": "ganti_kampas_rem"},
    {"id": "R24", "if": ["indikasi_kampas_habis"], "then": "ganti_kampas_rem"},

    # ---- Goal: Ganti minyak rem
    {"id": "R25", "if": ["bln_minyak_rem_lewat"], "then": "ganti_minyak_rem"},
    {"id": "R26", "if": ["minyak_rem_keruh"],     "then": "ganti_minyak_rem"},

    # ---- Goal: Ganti air radiator (hanya motor berradiator)
    {"id": "R27", "if": ["motor_berradiator", "bln_coolant_lewat"],                    "then": "ganti_coolant"},
    {"id": "R28", "if": ["motor_berradiator", "mesin_overheat", "coolant_berkurang"],  "then": "ganti_coolant"},
]

### 2.5 Daftar Goal (Jenis Servis)

In [6]:
# 1e. Daftar GOAL beserta deskripsi tindakan servis
# ---------------------------------------------------------------------
GOALS = {
    "ganti_oli_mesin":    ("Ganti Oli Mesin", "Ganti oli sesuai spesifikasi (mis. SAE 10W-30, JASO MB) dan bersihkan saringan kasa oli."),
    "ganti_oli_gardan":   ("Ganti Oli Gardan (Final Drive)", "Ganti oli gardan ±120 ml sesuai spesifikasi pabrikan, cek seal bila rembes."),
    "ganti_busi":         ("Ganti Busi", "Ganti busi sesuai tipe pabrikan, cek celah elektroda dan kondisi koil."),
    "ganti_filter_udara": ("Ganti Filter Udara", "Ganti saringan udara (tipe viscous tidak boleh dibersihkan dengan semprotan angin)."),
    "servis_cvt":         ("Servis / Pembersihan CVT", "Bersihkan debu kampas di ruang CVT, periksa roller, slider, dan mangkok kopling."),
    "ganti_vbelt":        ("Ganti V-Belt", "Ganti V-belt; periksa juga roller dan pulley karena biasanya aus bersamaan."),
    "servis_injeksi":     ("Servis Injektor & Throttle Body", "Bersihkan injektor dan throttle body dengan injector cleaner, lalu reset ECU."),
    "ganti_kampas_rem":   ("Ganti Kampas Rem", "Ganti kampas rem depan/belakang, periksa permukaan cakram/tromol."),
    "ganti_minyak_rem":   ("Ganti Minyak Rem", "Kuras dan ganti minyak rem (DOT 3/DOT 4), lalu buang angin (bleeding)."),
    "ganti_coolant":      ("Ganti Air Radiator (Coolant)", "Kuras dan isi ulang coolant, periksa selang dan tutup radiator."),
}

NAMA_FAKTA_ANTARA = {
    "indikasi_pengapian_lemah": "Indikasi pengapian lemah",
    "indikasi_masalah_cvt":     "Indikasi masalah pada CVT",
    "indikasi_injeksi_kotor":   "Indikasi injektor/throttle body kotor",
    "indikasi_kampas_habis":    "Indikasi kampas rem habis",
}

print(f"Basis pengetahuan dimuat: {len(GOALS)} goal, {len(RULES)} rule, "
      f"{len(FAKTA_DASAR)} fakta dasar, {len(PARAMETER)} parameter numerik")

Basis pengetahuan dimuat: 10 goal, 28 rule, 31 fakta dasar, 11 parameter numerik


## 3. Mesin Inferensi Backward Chaining

Inti metode ada di fungsi **`_buktikan()`** yang bersifat **rekursif**. Langkah 1–7 diberi komentar di dalam kode:

| Langkah | Proses |
|---|---|
| 1 | Cek *working memory*, jika sudah diketahui langsung dipakai |
| 2 | Cari rule yang THEN-nya = goal |
| 3 | Tidak ada rule → fakta dasar → **tanya user** |
| 4 | **Iterasi** setiap rule kandidat (OR) |
| 5 | Setiap premis jadi **sub-goal**, dibuktikan rekursif (AND) |
| 6 | Semua premis benar → goal **TERBUKTI**, iterasi berhenti |
| 7 | Semua rule gagal → goal **TIDAK terbukti** |

Fungsi **`jelaskan()`** adalah *explanation facility* untuk menjawab pertanyaan "kenapa servis ini direkomendasikan?".

In [7]:
class BackwardChaining:
    def __init__(self, rules, fakta_dasar, parameter, interval, penjawab):
        self.rules = rules
        self.fakta_dasar = fakta_dasar
        self.parameter = parameter
        self.interval = interval
        self.penjawab = penjawab        # objek yang bertugas menanyai pengguna
        self.memori = {}                # working memory: fakta -> True/False
        self.nilai_param = {}           # cache jawaban numerik
        self.bukti = {}                 # goal -> rule yang berhasil membuktikannya
        self.trace = []                 # catatan proses inferensi
        self.jalur = []                 # jalur penelusuran aktif: goal -> rule -> premis ...
        self.goal_aktif = None          # goal utama yang sedang dibuktikan

    def log(self, depth, teks):
        self.trace.append("    " * depth + teks)

    # -----------------------------------------------------------------
    # Fungsi inti backward chaining (rekursif)
    # -----------------------------------------------------------------
    def buktikan(self, goal, depth=0):
        # Catat goal ke jalur penelusuran (dipakai GUI untuk visualisasi)
        self.jalur.append(goal)
        hasil = self._buktikan(goal, depth)
        self.jalur.pop()
        return hasil

    def _buktikan(self, goal, depth):
        # [LANGKAH 1] Cek working memory: jika goal sudah pernah dibuktikan
        # (benar/salah), pakai hasilnya agar tidak bertanya dua kali.
        if goal in self.memori:
            self.log(depth, f"- '{goal}' sudah diketahui = {self.memori[goal]}")
            return self.memori[goal]

        # [LANGKAH 2] Cari semua rule yang KESIMPULANNYA (THEN) adalah goal.
        rule_kandidat = [r for r in self.rules if r["then"] == goal]

        # [LANGKAH 3] Tidak ada rule yang menyimpulkan goal -> goal adalah
        # FAKTA DASAR, sehingga nilainya diperoleh dengan bertanya ke user.
        if not rule_kandidat:
            hasil = self.tanya_fakta(goal)
            self.memori[goal] = hasil
            self.log(depth, f"- Fakta dasar '{goal}' = {hasil}")
            return hasil

        # [LANGKAH 4] ITERASI BACKWARD CHAINING:
        # Coba setiap rule kandidat satu per satu (hubungan OR).
        for rule in rule_kandidat:
            self.log(depth, f"> Mencoba {rule['id']}: IF {' AND '.join(rule['if'])} THEN {goal}")
            terpenuhi = True
            self.jalur.append(rule["id"])

            # [LANGKAH 5] Setiap premis menjadi SUB-GOAL baru dan dibuktikan
            # secara rekursif (mundur dari kesimpulan menuju fakta).
            for premis in rule["if"]:
                if not self.buktikan(premis, depth + 1):
                    # Satu premis gagal -> rule gagal (hubungan AND),
                    # premis berikutnya tidak perlu ditanyakan.
                    terpenuhi = False
                    self.log(depth, f"x {rule['id']} gagal (premis '{premis}' tidak terpenuhi)")
                    break
            self.jalur.pop()

            # [LANGKAH 6] Semua premis terbukti -> goal TERBUKTI.
            # Iterasi dihentikan, rule kandidat lain tidak perlu dicoba.
            if terpenuhi:
                self.memori[goal] = True
                self.bukti[goal] = rule
                self.log(depth, f"✓ {rule['id']} berhasil -> '{goal}' TERBUKTI")
                return True

        # [LANGKAH 7] Semua rule kandidat gagal -> goal TIDAK terbukti.
        self.memori[goal] = False
        self.log(depth, f"x Semua rule untuk '{goal}' gagal -> TIDAK terbukti")
        return False

    # -----------------------------------------------------------------
    # Mengambil nilai fakta dasar dari pengguna
    # -----------------------------------------------------------------
    def tanya_fakta(self, fakta):
        info = self.fakta_dasar[fakta]
        if info["tipe"] == "ya_tidak":
            return self.penjawab.ya_tidak(fakta, info["tanya"])

        # Fakta numerik: parameter cukup ditanya sekali, lalu dibandingkan
        # dengan batas interval servis.
        param = info["param"]
        if param not in self.nilai_param:
            self.nilai_param[param] = self.penjawab.angka(param, self.parameter[param])
        return self.nilai_param[param] >= self.interval[info["batas"]]

    # -----------------------------------------------------------------
    # Jalankan backward chaining untuk seluruh goal
    # -----------------------------------------------------------------
    def jalankan(self, daftar_goal):
        hasil = []
        for goal in daftar_goal:
            self.goal_aktif = goal
            self.log(0, f"\n=== GOAL: {goal} ===")
            if self.buktikan(goal):
                hasil.append(goal)
        self.goal_aktif = None
        return hasil

    # -----------------------------------------------------------------
    # Penjelasan (explanation facility): kenapa goal direkomendasikan
    # -----------------------------------------------------------------
    def jelaskan(self, goal):
        rule = self.bukti.get(goal)
        if not rule:
            return []
        alasan = []
        for premis in rule["if"]:
            if premis in self.bukti:                       # premis = sub-goal
                alasan.append(f"{NAMA_FAKTA_ANTARA.get(premis, premis)} ({self.bukti[premis]['id']}):")
                alasan += ["    > " + a for a in self.jelaskan(premis)]
            elif self.fakta_dasar.get(premis, {}).get("tipe") == "numerik":
                info = self.fakta_dasar[premis]
                nilai = self.nilai_param[info["param"]]
                batas = self.interval[info["batas"]]
                satuan = "bulan" if info["param"].startswith("bln") else "km"
                komponen = self.parameter[info["param"]].split("terakhir ")[1].rstrip("?")
                alasan.append(f"Sudah {nilai:,} {satuan} sejak terakhir {komponen.lower()} "
                              f"(batas interval {batas:,} {satuan})".replace(",", "."))
            else:
                alasan.append(f"Ya: {self.fakta_dasar[premis]['tanya']}")
        return alasan

## 4. Pengujian dengan Skenario Demo

Jawaban pengguna diisi otomatis (`JAWABAN_DEMO`) supaya proses inferensinya bisa langsung dilihat. Perhatikan:
- Pertanyaan **bulan oli mesin** muncul karena R06 (km) gagal, lalu sistem mencoba R07.
- Pertanyaan **indikator oli** tidak pernah muncul karena R07 sudah berhasil.
- Pertanyaan **coolant** tidak muncul karena motor tidak berradiator (premis pertama R27/R28 gagal).

In [8]:
class PenjawabKonsol:
    """Menanyakan fakta ke pengguna lewat terminal."""

    def ya_tidak(self, fakta, pertanyaan):
        while True:
            jwb = input(f"  [?] {pertanyaan} (y/t): ").strip().lower()
            if jwb in ("y", "ya"):
                return True
            if jwb in ("t", "tidak", "n"):
                return False
            print("      Jawab dengan 'y' atau 't'.")

    def angka(self, param, pertanyaan):
        while True:
            jwb = input(f"  [?] {pertanyaan} ").strip().replace(".", "")
            if jwb.isdigit():
                return int(jwb)
            print("      Masukkan angka bulat, contoh: 4500")


class PenjawabDemo:
    """Jawaban otomatis untuk demo/pengujian (tanpa input manual)."""

    def __init__(self, jawaban):
        self.jawaban = jawaban

    def ya_tidak(self, fakta, pertanyaan):
        nilai = self.jawaban.get(fakta, False)
        print(f"  [?] {pertanyaan} (y/t): {'y' if nilai else 't'}")
        return nilai

    def angka(self, param, pertanyaan):
        nilai = self.jawaban.get(param, 0)
        print(f"  [?] {pertanyaan} {nilai}")
        return nilai


JAWABAN_DEMO = {
    "km_oli_mesin": 3200, "bln_oli_mesin": 4,
    "km_oli_gardan": 5000, "bln_oli_gardan": 10,
    "km_busi": 6000, "mesin_susah_hidup": True, "mesin_brebet": True,
    "km_filter": 9000, "sering_jalan_berdebu": True,
    "km_cvt": 5000, "tarikan_berat": True, "bunyi_cvt": True,
    "km_vbelt": 17000,
    "km_injeksi": 6000, "boros_bensin": False,
    "kampas_batas_aus": False, "rem_berbunyi": True, "rem_kurang_pakem": False,
    "bln_minyak_rem": 12, "minyak_rem_keruh": False,
    "motor_berradiator": False,
}


def tampilkan_hasil(engine, rekomendasi, tampil_trace=True):
    garis = "=" * 65
    if tampil_trace:
        print(f"\n{garis}\n PROSES INFERENSI BACKWARD CHAINING\n{garis}")
        print("\n".join(engine.trace))

    print(f"\n{garis}\n HASIL REKOMENDASI SERVIS\n{garis}")
    if not rekomendasi:
        print("Motor dalam kondisi baik. Belum ada servis yang perlu dilakukan.")
    for i, goal in enumerate(rekomendasi, 1):
        nama, tindakan = GOALS[goal]
        print(f"\n{i}. {nama}  [{engine.bukti[goal]['id']}]")
        print(f"   Tindakan : {tindakan}")
        print("   Alasan   :")
        for a in engine.jelaskan(goal):
            print(f"     {'' if a.startswith('    >') else '- '}{a}")

    tidak = [GOALS[g][0] for g in GOALS if g not in rekomendasi]
    if tidak:
        print(f"\nBelum perlu: {', '.join(tidak)}")
    print("\n* Rekomendasi bersifat panduan. Ikuti buku pedoman pemilik motor Anda.")

In [9]:
engine_demo = BackwardChaining(RULES, FAKTA_DASAR, PARAMETER, INTERVAL, PenjawabDemo(JAWABAN_DEMO))
rekomendasi_demo = engine_demo.jalankan(list(GOALS))
tampilkan_hasil(engine_demo, rekomendasi_demo)

  [?] Berapa km sejak terakhir ganti OLI MESIN? 3200
  [?] Berapa bulan sejak terakhir ganti OLI MESIN? 4
  [?] Berapa km sejak terakhir ganti OLI GARDAN? 5000
  [?] Berapa bulan sejak terakhir ganti OLI GARDAN? 10
  [?] Apakah ada rembesan oli di area gardan/roda belakang? (y/t): t
  [?] Berapa km sejak terakhir ganti BUSI? 6000
  [?] Apakah mesin susah dihidupkan? (y/t): y
  [?] Apakah mesin brebet/tersendat saat digas? (y/t): y
  [?] Berapa km sejak terakhir ganti FILTER UDARA? 9000
  [?] Apakah motor sering dipakai di jalan berdebu/proyek? (y/t): y
  [?] Berapa km sejak terakhir SERVIS/BERSIHKAN CVT? 5000
  [?] Apakah tarikan motor terasa berat/loyo? (y/t): y
  [?] Apakah terdengar bunyi kasar/mendengung dari area CVT? (y/t): y
  [?] Berapa km sejak terakhir ganti V-BELT? 17000
  [?] Berapa km sejak terakhir SERVIS INJEKSI/THROTTLE BODY? 6000
  [?] Apakah konsumsi bensin terasa lebih boros? (y/t): t
  [?] Apakah kampas rem sudah mencapai garis/indikator batas aus (±1 mm)? (y/t): t


> 💡 **Mode teks interaktif (opsional):** hapus tanda `#` di cell berikut untuk konsultasi lewat `input()`.

In [10]:
# engine_teks = BackwardChaining(RULES, FAKTA_DASAR, PARAMETER, INTERVAL, PenjawabKonsol())
# tampilkan_hasil(engine_teks, engine_teks.jalankan(list(GOALS)))

## 5. Pengujian *Black Box*

Beberapa skenario kondisi motor diuji untuk memastikan hasil sistem sesuai dengan harapan pakar. Kolom **Jumlah Pertanyaan** menunjukkan efisiensi backward chaining dibandingkan jika semua fakta ditanyakan.

In [11]:
class PenjawabDiam:
    """Penjawab otomatis tanpa output. Fakta yang tidak diisi = Tidak / 0."""
    def __init__(self, jawaban):
        self.jawaban, self.ditanya = jawaban, 0
    def ya_tidak(self, fakta, pertanyaan):
        self.ditanya += 1
        return self.jawaban.get(fakta, False)
    def angka(self, param, pertanyaan):
        self.ditanya += 1
        return self.jawaban.get(param, 0)


SKENARIO = [
    ("Motor baru, tidak ada keluhan", {}, []),
    ("Oli mesin sudah 4.500 km", {"km_oli_mesin": 4500}, ["ganti_oli_mesin"]),
    ("Oli mesin 2.000 km tapi sudah 4 bulan", {"km_oli_mesin": 2000, "bln_oli_mesin": 4}, ["ganti_oli_mesin"]),
    ("Mesin susah hidup & brebet", {"mesin_susah_hidup": True, "mesin_brebet": True}, ["ganti_busi"]),
    ("Motor gredek, V-belt 17.000 km", {"gredek": True, "km_vbelt": 17000}, ["servis_cvt", "ganti_vbelt"]),
    ("Rem bunyi & kurang pakem", {"rem_berbunyi": True, "rem_kurang_pakem": True}, ["ganti_kampas_rem"]),
    ("Motor radiator, coolant 40 bulan", {"motor_berradiator": True, "bln_coolant": 40}, ["ganti_coolant"]),
    ("Overheat tapi motor tanpa radiator", {"mesin_overheat": True, "coolant_berkurang": True}, []),
    ("Servis besar 24.000 km", {"km_oli_mesin": 24000, "km_oli_gardan": 24000, "km_busi": 24000,
                                "km_filter": 24000, "km_cvt": 24000, "km_vbelt": 24000, "km_injeksi": 24000},
     ["ganti_oli_mesin", "ganti_oli_gardan", "ganti_busi", "ganti_filter_udara",
      "servis_cvt", "ganti_vbelt", "servis_injeksi"]),
]

TOTAL_KEMUNGKINAN = len(PARAMETER) + sum(1 for f in FAKTA_DASAR.values() if f["tipe"] == "ya_tidak")
baris = []
for i, (nama, jawaban, harapan) in enumerate(SKENARIO, 1):
    pj = PenjawabDiam(jawaban)
    eng = BackwardChaining(RULES, FAKTA_DASAR, PARAMETER, INTERVAL, pj)
    hasil = eng.jalankan(list(GOALS))
    baris.append({
        "No": i,
        "Skenario": nama,
        "Hasil Sistem": ", ".join(f"{GOALS[g][0]} ({eng.bukti[g]['id']})" for g in hasil) or "Tidak ada servis",
        "Jumlah Pertanyaan": f"{pj.ditanya} dari {TOTAL_KEMUNGKINAN}",
        "Status": "✅ Sesuai" if sorted(hasil) == sorted(harapan) else "❌ Tidak sesuai",
    })

df_uji = pd.DataFrame(baris)
sesuai = (df_uji["Status"] == "✅ Sesuai").sum()
print(f"Akurasi pengujian: {sesuai}/{len(df_uji)} skenario sesuai ({sesuai / len(df_uji) * 100:.0f}%)")
df_uji.style.hide(axis="index").set_properties(**{"text-align": "left"})

Akurasi pengujian: 9/9 skenario sesuai (100%)


No,Skenario,Hasil Sistem,Jumlah Pertanyaan,Status
1,"Motor baru, tidak ada keluhan",Tidak ada servis,22 dari 29,✅ Sesuai
2,Oli mesin sudah 4.500 km,Ganti Oli Mesin (R06),19 dari 29,✅ Sesuai
3,Oli mesin 2.000 km tapi sudah 4 bulan,Ganti Oli Mesin (R07),20 dari 29,✅ Sesuai
4,Mesin susah hidup & brebet,Ganti Busi (R14),23 dari 29,✅ Sesuai
5,"Motor gredek, V-belt 17.000 km","Servis / Pembersihan CVT (R18), Ganti V-Belt (R20)",22 dari 29,✅ Sesuai
6,Rem bunyi & kurang pakem,Ganti Kampas Rem (R24),23 dari 29,✅ Sesuai
7,"Motor radiator, coolant 40 bulan",Ganti Air Radiator (Coolant) (R27),23 dari 29,✅ Sesuai
8,Overheat tapi motor tanpa radiator,Tidak ada servis,22 dari 29,✅ Sesuai
9,Servis besar 24.000 km,"Ganti Oli Mesin (R06), Ganti Oli Gardan (Final Drive) (R10), Ganti Busi (R13), Ganti Filter Udara (R15), Servis / Pembersihan CVT (R17), Ganti V-Belt (R19), Servis Injektor & Throttle Body (R21)",12 dari 29,✅ Sesuai


## 6. Simpan Basis Pengetahuan

Seluruh basis pengetahuan (interval, parameter, fakta, rule, goal) disimpan ke file **`basis_pengetahuan_servis.sav`** menggunakan `pickle`.
File ini kemudian di-*load* oleh aplikasi web `app.py`, sama seperti model machine learning yang disimpan lalu dipakai aplikasi.

In [12]:
basis_pengetahuan = {
    "INTERVAL": INTERVAL,
    "PARAMETER": PARAMETER,
    "FAKTA_DASAR": FAKTA_DASAR,
    "RULES": RULES,
    "GOALS": GOALS,
    "NAMA_FAKTA_ANTARA": NAMA_FAKTA_ANTARA,
    "JAWABAN_DEMO": JAWABAN_DEMO,
}

filename = "basis_pengetahuan_servis.sav"
pickle.dump(basis_pengetahuan, open(filename, "wb"))
print(f"Basis pengetahuan disimpan ke '{filename}'")

Basis pengetahuan disimpan ke 'basis_pengetahuan_servis.sav'


In [13]:
# Cek: load ulang file .sav lalu uji dengan skenario demo
kb = pickle.load(open("basis_pengetahuan_servis.sav", "rb"))
print("Isi file:", list(kb.keys()))

uji = BackwardChaining(kb["RULES"], kb["FAKTA_DASAR"], kb["PARAMETER"], kb["INTERVAL"], PenjawabDiam(kb["JAWABAN_DEMO"]))
hasil = uji.jalankan(list(kb["GOALS"]))
print("Hasil dari file .sav:", [kb["GOALS"][g][0] for g in hasil])

Isi file: ['INTERVAL', 'PARAMETER', 'FAKTA_DASAR', 'RULES', 'GOALS', 'NAMA_FAKTA_ANTARA', 'JAWABAN_DEMO']
Hasil dari file .sav: ['Ganti Oli Mesin', 'Ganti Busi', 'Ganti Filter Udara', 'Servis / Pembersihan CVT', 'Ganti V-Belt']


## 7. Menjalankan Aplikasi Web (`app.py`)

Aplikasi interaktif dibuat terpisah di file **`app.py`** menggunakan **Streamlit**. Mesin inferensi yang dipakai sama dengan notebook ini.

**Cara menjalankan** (buka Terminal / Anaconda Prompt di folder projek):
```bash
pip install streamlit
streamlit run app.py
```
Browser akan terbuka otomatis di `http://localhost:8501`.

**Fitur aplikasi:**
- **Jalankan Demo**: konsultasi berjalan otomatis (cocok untuk presentasi)
- **Jalur penelusuran** (goal → rule → sub-goal → fakta) di setiap pertanyaan
- **Log inferensi LIVE** berwarna
- **Status 10 goal** di sidebar (CEK / PERLU / AMAN)
- Tombol **Kembali** & **Mulai Ulang**
- Hasil: rekomendasi + alasan, proses inferensi, tabel jawaban, dan **unduh laporan .txt**

**Teknik penghubung Streamlit ↔ mesin inferensi:** Streamlit menjalankan ulang script setiap ada klik. Jawaban disimpan di `st.session_state`, lalu inferensi dijalankan dari awal. Saat mesin butuh fakta yang belum dijawab, ia melempar `PerluJawaban` dan aplikasi menampilkan pertanyaan tersebut.

## 8. Kesimpulan

1. Sistem pakar berhasil merekomendasikan **10 jenis servis berkala** motor matic menggunakan **28 rule** dan metode **Backward Chaining**.
2. Backward chaining bekerja **mundur dari goal ke fakta**, sehingga sistem **hanya menanyakan fakta yang relevan**. Pada pengujian, jumlah pertanyaan selalu lebih sedikit dari total kemungkinan pertanyaan.
3. Adanya **sub-goal** (misal *indikasi masalah CVT*) membuat satu fakta antara bisa dipakai ulang oleh beberapa goal (servis CVT dan ganti V-belt) tanpa bertanya ulang, berkat *working memory*.
4. *Explanation facility* membuat setiap rekomendasi dapat dipertanggungjawabkan karena alasan dan rule pembuktinya ditampilkan.
5. Hasil pengujian *black box* menunjukkan seluruh skenario menghasilkan rekomendasi yang sesuai dengan harapan.

### Referensi
1. Pratama, A. G., Rizky, R., Yunita, A. M., & Wardah, N. N. (2020). Implementasi Metode Backward Chaining untuk Diagnosa Kerusakan Motor Matic Injection. *Explore: Jurnal Sistem Informasi dan Telematika*, 11(2), 91–96. https://doi.org/10.36448/jsit.v11i2.1515
2. Ilyas, M., Hutagalung, J. E., & Suparmadi. (2022). Penerapan Metode Forward Chaining Pada Sistem Pakar Untuk Diagnosa Kerusakan Sepeda Motor Berbasis Web. *Building of Informatics, Technology and Science (BITS)*, 4(2), 932–942. https://doi.org/10.47065/bits.v4i2.2163
3. Akhsa, A. C. D. (2024). Sistem Pakar Diagnosa Kerusakan Motor Matic Menggunakan Metode Forward Chaining. *MALCOM: Indonesian Journal of Machine Learning and Computer Science*, 4(2), 452–462. https://doi.org/10.57152/malcom.v4i2.1189
4. PT Astra Honda Motor. *Buku Pedoman Pemilik Honda Vario 125*.